# Langlands 03 : la chaîne de modularité — de l'équation de Fermat à Wiles

Troisième carnet de la série **Langlands** ([Epic #17969](https://github.com/jsboige/CoursIA/issues/17969)). Le [carnet 01](01-formes-modulaires-sl2z-hecke.ipynb) a posé le socle — q-expansions, opérateurs de Hecke, le pont $\Gamma_0(11)$ ↔ courbe 11a1 — et le [carnet 02](02-monstrous-moonshine-invariant-j.ipynb) a suivi l'invariant $j$ jusqu'au groupe Monstre. Ici, on remonte la chaîne complète qui a mené à la preuve du **théorème de Fermat** : une solution hypothétique devient une courbe elliptique (Frey), la courbe serait modulaire (Taniyama–Shimura–Weil), la forme associée descendrait en niveau 2 (Serre, Ribet) — et le niveau 2 de poids 2 est **vide**.

Le geste de la série s'applique au théorème le plus célèbre des mathématiques : chaque maillon est **calculé dans une sortie** (discriminant, tables $a_p$, classes de cosets) ou **démontré dans le lake** [`hecke_lean/Hecke/FltRoute.lean`](../hecke_lean/Hecke/FltRoute.lean) — et les maillons que ni le calcul ni le lake ne couvrent sont dits **admis**, nommément.

**Prérequis** : carnet 01, sections 4 à 6 (opérateur de Hecke, $\Gamma_0(11)$, comptage de points).
**Durée** : environ 1 h. **Outil** : Python stdlib seul — arithmétique exacte, aucun flottant dans les identités.


## Plan

1. Le théorème et ses réductions — il ne reste que $n = 4$, $n = 3$ et les premiers $p \geq 5$
2. La courbe de Frey — transformer une solution hypothétique en courbe elliptique
3. La modularité — l'énoncé, et sa face calculée
4. La descente de Ribet — le niveau 2 vide
5. Le lake `hecke_lean` — démontré vs admis
6. Épilogue — ce que la chaîne a changé
7. Exercices


## 1. Le théorème et ses réductions

L'équation $a^n + b^n = c^n$ **n'a pas de solution en entiers strictement positifs pour $n > 2$**. Fermat l'écrit en marge de son Diophante vers 1637, avec cette précision d'avare : *une démonstration admirable que cette marge est trop étroite pour contenir*. Il faudra 358 ans.

Un principe de réduction divise le problème. Si $a^{km} + b^{km} = c^{km}$, alors $(a^k, b^k, c^k)$ est une solution pour l'exposant $m$ : **il suffit de traiter les exposants premiers et $n = 4$**. Fermat lui-même démontre $n = 4$ (descente infinie), Euler traite $n = 3$ — restent les premiers $p \geq 5$, exactement le terrain de Wiles.

Vérifions la réduction sur les premiers exposants :

In [1]:
def plus_petit_premier_impair_divisant(n):
    """Plus petit facteur premier impair de n (n >= 3, impair non premier compris)."""
    for d in range(3, n + 1, 2):
        if n % d == 0 and all(d % q for q in range(2, int(d ** 0.5) + 1)):
            return d
    return None

print(f"{'n':>3} | {'statut':>34}")
print("-" * 42)
for n in range(2, 25):
    if n == 2:
        verdict = "solutions (triplets pythagoriciens)"
    elif n == 4:
        verdict = "a traiter : Fermat, descente infinie"
    elif n == 3:
        verdict = "a traiter : Euler"
    elif n % 4 == 0:
        verdict = "reduit au cas n = 4"
    elif n >= 5 and all(n % q for q in range(2, int(n ** 0.5) + 1)):
        verdict = "a traiter : premier p >= 5 (Wiles)"
    else:
        p = plus_petit_premier_impair_divisant(n)
        verdict = f"reduit au cas n = {p}"
    print(f"{n:>3} | {verdict:>34}")
print()
print("Le theoreme entier se joue donc sur : {4, 3} et les premiers p >= 5.")

  n |                             statut
------------------------------------------
  2 | solutions (triplets pythagoriciens)
  3 |                  a traiter : Euler
  4 | a traiter : Fermat, descente infinie
  5 | a traiter : premier p >= 5 (Wiles)
  6 |                reduit au cas n = 3
  7 | a traiter : premier p >= 5 (Wiles)
  8 |                reduit au cas n = 4
  9 |                reduit au cas n = 3
 10 |                reduit au cas n = 5
 11 | a traiter : premier p >= 5 (Wiles)
 12 |                reduit au cas n = 4
 13 | a traiter : premier p >= 5 (Wiles)
 14 |                reduit au cas n = 7
 15 |                reduit au cas n = 3
 16 |                reduit au cas n = 4
 17 | a traiter : premier p >= 5 (Wiles)
 18 |                reduit au cas n = 3
 19 | a traiter : premier p >= 5 (Wiles)
 20 |                reduit au cas n = 4
 21 |                reduit au cas n = 3
 22 |               reduit au cas n = 11
 23 | a traiter : premier p >= 5 (Wiles)
 24 |      

La table se lit comme un entonnoir : tout exposant composé retombe sur un cas premier ou sur $n=4$. Les cas $n=3$ et $n=4$ sont classiques ; tout le reste du carnet concerne les premiers $p \geq 5$ — le seul terrain où la chaîne de modularité opère.

## 2. La courbe de Frey — transformer une solution hypothétique en courbe elliptique

L'idée de Gerhard Frey (1984) est un **contre-pied** : au lieu de chercher à prouver qu'il n'y a pas de solution, il en *suppose* une — $(a, b, c)$ premiers entre eux, $p \geq 5$ premier, $a^p + b^p = c^p$ — et en fabrique une courbe elliptique :

$$E_{\text{Frey}} : y^2 = x\,(x - a^p)(x + b^p).$$

Ses invariants sont** monstueusement contraints**. Posons $A = a^p$, $B = b^p$ : les racines du membre de droite sont $0$, $A$ et $-B$, donc le discriminant de la cubique vaut $A^2 B^2 (A+B)^2$ — et pour un contre-exemple, $A + B = c^p$ :

$$\Delta = 16\,(abc)^{2p}.$$

Chaque premier du discriminant y apparaît avec l'exposant $2p$ : la courbe serait *minimalement ramifiée* partout, **semistable**. C'est cette régularité extrême qui la rendra impossible.

Pour *calculer* sur une courbe de Frey, un problème : il n'existe pas de contre-exemple. Le modèle jouet honnête est l'exposant $n = 2$, là où les solutions **existent** — le triplet pythagoricien $(3, 4, 5)$ donne $A = 9$, $B = 16$, une vraie courbe $y^2 = x(x-9)(x+16)$ sur laquelle tout se calcule.

In [2]:
from fractions import Fraction
import math

def invariants_frey(A, B):
    """c4, c6, Delta de y^2 = x(x - A)(x + B), par les invariants b de Silverman."""
    a2, a4, a6 = B - A, -A * B, 0          # y^2 = x^3 + (B-A)x^2 - AB x
    b2, b4, b6 = 4 * a2, 2 * a4, 4 * a6
    b8 = 4 * a2 * a6 - a4 * a4
    c4 = b2 * b2 - 24 * b4
    c6 = -b2 ** 3 + 36 * b2 * b4
    delta = -b2 * b2 * b8 - 8 * b4 ** 3 - 27 * b6 * b6 + 9 * b2 * b4 * b6
    return c4, c6, delta

A, B = 3 ** 2, 4 ** 2                       # triplet (3, 4, 5), exposant n = 2
c4, c6, delta = invariants_frey(A, B)
delta_racines = 16 * (A * B * (A + B)) ** 2 # 16 * produit des (racine_i - racine_j)^2

print(f"modele jouet : y^2 = x(x - {A})(x + {B})")
print(f"c4 = {c4}   (= 16*(A^2 + AB + B^2) = {16 * (A*A + A*B + B*B)})")
print(f"c6 = {c6}")
print(f"Delta par les invariants b = {delta}")
print(f"Delta par les racines 0, A, -B = {delta_racines}")
assert delta == delta_racines
assert c4 == 16 * (A * A + A * B + B * B)
assert c4 ** 3 - c6 ** 2 == 1728 * delta
print("identites verifiees : Delta = 16*(A*B*(A+B))^2, c4 = 16*(A^2+AB+B^2), c4^3 - c6^2 = 1728*Delta")
print()
print("Pour un vrai contre-exemple (a, b, c, p) : Delta = 16 * (a*b*c)^(2p).")
print("Structurellement : Delta = 16 * (A * B * (A + B))^2 quel que soit (A, B).")

modele jouet : y^2 = x(x - 9)(x + 16)
c4 = 7696   (= 16*(A^2 + AB + B^2) = 7696)
c6 = -312256
Delta par les invariants b = 207360000
Delta par les racines 0, A, -B = 207360000
identites verifiees : Delta = 16*(A*B*(A+B))^2, c4 = 16*(A^2+AB+B^2), c4^3 - c6^2 = 1728*Delta

Pour un vrai contre-exemple (a, b, c, p) : Delta = 16 * (a*b*c)^(2p).
Structurellement : Delta = 16 * (A * B * (A + B))^2 quel que soit (A, B).


Les deux routes — invariants $b$ de la forme générale, produit des différences de racines carrées — tombent d'accord dans la sortie : c'est l'identité $\Delta = 16\,A^2B^2(A+B)^2$, **calculée** et non citée. Le $c_4$ se factorise aussi proprement : $c_4 = 16\,(A^2 + AB + B^2)$, et sur un contre-exemple $A^2 + AB + B^2 = (A+B)^2 - AB = c^{2p} - (ab)^p$ — tous les invariants de la courbe de Frey s'écrivent en $a, b, c$.

In [3]:
j = Fraction(c4 ** 3, delta)
print(f"j = c4^3 / Delta = {c4}^3 / {delta}")
print(f"j = {j.numerator} / {j.denominator}")
print(f"j = {float(j):.3f}   (valeur approchee, lecture seule)")

j = c4^3 / Delta = 7696^3 / 207360000
j = 111284641 / 50625
j = 2198.215   (valeur approchee, lecture seule)


**Lecture.** Un $j$ rationnel — toute courbe elliptique sur $\mathbb{Q}$ en a un rationnel — mais dont le dénominateur ne contient, par la structure $\Delta = 16(abc)^{2p}$, que les premiers de $abc$, chacun à une puissance énorme. La courbe de Frey d'un contre-exemple serait une courbe elliptique d'une régularité arithmétique sans équivalent : c'est elle qui va entrer en collision avec la théorie des formes modulaires.

## 3. La modularité — l'énoncé, et sa face calculée

La conjecture de Taniyama–Shimura (1955), précisée par Weil (1967), relie les deux mondes que les carnets 01 et 02 ont parcourus séparément :

> **Toute courbe elliptique $E$ sur $\mathbb{Q}$, de conducteur $N$, est modulaire : il existe une forme parabolique $f$ de poids 2 pour $\Gamma_0(N)$, à coefficients entiers, telle que $L(E, s) = L(f, s)$.**

Wiles (avec Taylor) la démontre en 1995 pour les courbes **semistables** — exactement la classe de la courbe de Frey ; Breuil–Conrad–Diamond–Taylor la démontrent en entier en 2001. La face calculée de l'énoncé, établie au carnet 01 section 6, tient en une table : le $p$-ième coefficient $a_p$ de $f$ vaut $p + 1 - \#E(\mathbb{F}_p)$, compté indépendamment par géométrie. La courbe 11a1 — $y^2 + y = x^3 - x^2 - 10x - 20$, conducteur 11 — et la forme $\eta^2\eta_{11}^2 = q\prod(1-q^n)^2(1-q^{11n})^2$ du carnet 01 en donnent la version calculable :

In [4]:
def points_affines_E11(p):
    """Solutions affines de y^2 + y = x^3 - x^2 - 10x - 20 dans F_p x F_p."""
    return sum(1 for x in range(p) for y in range(p)
               if (y * y + y - (x ** 3 - x ** 2 - 10 * x - 20)) % p == 0)

def a_p_courbe(p):
    """a_p = p + 1 - #E(F_p), point a l'infini inclus."""
    return p + 1 - (points_affines_E11(p) + 1)

M = 40                                       # profondeur des q-expansions

def mul(u, v):
    """Produit de deux series tronquees a M, coefficients entiers exacts."""
    w = [0] * (M + 1)
    for i, a in enumerate(u):
        if a:
            for j, b in enumerate(v[:M + 1 - i]):
                w[i + j] += a * b
    return w

def un_moins_qn(n):
    """(1 - q^n) tronque."""
    base = [1] + [0] * M
    if n <= M:
        base[n] -= 1
    return base

produit = [1] + [0] * M                      # prod (1-q^n)^2 (1-q^{11n})^2
for n in range(1, M + 1):
    produit = mul(produit, mul(un_moins_qn(n), un_moins_qn(n)))
    if 11 * n <= M:
        produit = mul(produit, mul(un_moins_qn(11 * n), un_moins_qn(11 * n)))
f11 = [0] + produit[:M]                      # multiplication par q : decalage

premiers = (2, 3, 5, 7, 13, 17)
print(f"{'p':>3} | {'a_p geometrie':>13} | {'c_p forme eta^2 eta_11^2':>26}")
ok = True
for p in premiers:
    g, an = a_p_courbe(p), f11[p]
    ok = ok and (g == an)
    print(f"{p:>3} | {g:>13} | {an:>26}", "" if g == an else "  <-- ECART")
assert ok
print()
print("Deux calculs independants, une seule table : etre modulaire, c'est ceci.")
print("(p = 11, le conducteur, est exclu : reduction multiplicative, la table")
print(" se lit sur les bons premiers.)")

  p | a_p geometrie |   c_p forme eta^2 eta_11^2
  2 |            -2 |                         -2 
  3 |            -1 |                         -1 
  5 |             1 |                          1 
  7 |            -2 |                         -2 
 13 |             4 |                          4 
 17 |            -2 |                         -2 

Deux calculs independants, une seule table : etre modulaire, c'est ceci.
(p = 11, le conducteur, est exclu : reduction multiplicative, la table
 se lit sur les bons premiers.)


**Lecture.** La colonne de gauche compte des points sur des corps finis ; celle de droite multiplie des séries formelles. Aucun théorème ne les relie *dans le calcul* — et elles coïncident. C'est cette table que la modularité promet pour **toute** courbe elliptique sur $\mathbb{Q}$, y compris — par l'absurde — l'hypothétique courbe de Frey. En termes de fonctions $L$ : $L(E, s) = \sum a_n n^{-s} = L(f, s)$ ; les zéros de cette fonction, leur espacement et leurs statistiques, sont le sujet du carnet [`Serre100/07`](../Serre100/07-zeros-fonctions-l-gaps-gue.ipynb).

## 4. La descente de Ribet — le niveau 2 vide

Supposons la courbe de Frey modulaire. Elle serait associée à une forme de poids 2, de niveau le conducteur $N$ — lequel, par la structure $\Delta = 16(abc)^{2p}$, ne divise que le produit $abc$ (avec une petite correction pour le premier 2). Ken Ribet, démontrant en 1986 la conjecture $\varepsilon$ de Serre, fait mieux que constater : il **détruit** tous les premiers du niveau un à un. Chaque premier $\ell \mid N$ peut être « abaissé » — la forme vit aussi en niveau $N/\ell$ — jusqu'à ne rester que le squelette 2-adique : la forme de la courbe de Frey vivrait en **niveau 2**.

Or le niveau 2 de poids 2 est **vide**. Le groupe $\Gamma_0(2)$ — matrices de $SL_2(\mathbb{Z})$ avec $c \equiv 0 \pmod 2$ — est d'indice 3 dans $SL_2(\mathbb{Z})$, et la courbe modulaire $X_0(2)$ est de genre 0 : aucune forme parabolique de poids 2 n'y vit. L'indice, lui, se **calcule** :

In [5]:
def produit_mat(a, b):
    """Produit de deux matrices 2x2 a coefficients entiers."""
    return [[sum(a[i][k] * b[k][j] for k in range(2)) for j in range(2)] for i in range(2)]

def premiere_colonne_mod2(m):
    """Premiere colonne de m, reduite modulo 2 : (a mod 2, c mod 2)."""
    return (m[0][0] % 2, m[1][0] % 2)

I = [[1, 0], [0, 1]]                         # identite
S = [[0, -1], [1, 0]]                        # inversion tau -> -1/tau
T = [[1, 1], [0, 1]]                         # translation tau -> tau + 1
TS = produit_mat(T, S)

print("Gamma_0(2) = {matrices de SL_2(Z) dont c est pair}")
print("L'application  classe a gauche -> premiere colonne mod 2  est injective :")
print("deux representants de meme colonne diffèrent d'une matrice de Gamma_0(2).")
print()
print(f"{'representant':>14} | {'matrice':>18} | {'1re colonne mod 2':>18}")
for nom, m in (("I", I), ("S", S), ("TS", TS)):
    mat_txt = f"[[{m[0][0]}, {m[0][1]}], [{m[1][0]}, {m[1][1]}]]"
    print(f"{nom:>14} | {mat_txt:>18} | {str(premiere_colonne_mod2(m)):>18}")
print()
vecteurs = {premiere_colonne_mod2(m) for m in (I, S, TS)}
print(f"colonnes obtenues : {sorted(vecteurs)}")
print("ce sont exactement les 3 vecteurs non nuls de (Z/2)^2")
assert vecteurs == {(1, 0), (0, 1), (1, 1)}
print()
print("3 classes pour les 3 vecteurs non nuls de (Z/2)^2 : indice [SL_2(Z) : Gamma_0(2)] = 3.")

Gamma_0(2) = {matrices de SL_2(Z) dont c est pair}
L'application  classe a gauche -> premiere colonne mod 2  est injective :
deux representants de meme colonne diffèrent d'une matrice de Gamma_0(2).

  representant |            matrice |  1re colonne mod 2
             I |   [[1, 0], [0, 1]] |             (1, 0)
             S |  [[0, -1], [1, 0]] |             (0, 1)
            TS |  [[1, -1], [1, 0]] |             (1, 1)

colonnes obtenues : [(0, 1), (1, 0), (1, 1)]
ce sont exactement les 3 vecteurs non nuls de (Z/2)^2

3 classes pour les 3 vecteurs non nuls de (Z/2)^2 : indice [SL_2(Z) : Gamma_0(2)] = 3.


**Lecture.** L'indice 3 tombe d'une bijection : classes à gauche de $\Gamma_0(2)$ ↔ vecteurs non nuls de $(\mathbb{Z}/2)^2$, première colonne réduite modulo 2 — c'est exactement la bijection `cosetToProj` que le lake démontre injective **et** surjective. Avec deux pointes et un seul point elliptique d'ordre 2, la courbe $X_0(2)$ est de genre 0 : $\dim S_2(\Gamma_0(2)) = 0$, et ce vide est **démontré formellement** dans le lake — théorème `s2_gamma0_2_eq_zero`, exercice 4 de `FltRoute.lean`.

La chaîne se referme alors d'elle-même :

| Maillon | Contenu | Statut |
|---|---|---|
| 1. Frey | un contre-exemple donne une courbe semistable $E$ | démontré (lake : `freyCurve`, congruences mod 8) |
| 2. Modularité | $E$ est modulaire, de poids 2, niveau $N \mid 2abc$ | **théorème** (Wiles 1995, semistable) |
| 3. Descente | la forme vit en niveau 2 | **théorème** (Ribet 1986) |
| 4. Vide | $S_2(\Gamma_0(2)) = 0$ | démontré (lake : `s2_gamma0_2_eq_zero`) |
| 5. Contradiction | pas de forme ⇒ pas de courbe ⇒ pas de contre-exemple | démontré (lake : `flt_of_full_route`) |

Le lake assemble les maillons 1, 4 et 5 et **admet nommément** 2 et 3 — exactement ce que le carnet narré jusqu'ici.

## 5. Le lake `hecke_lean` — démontré vs admis

Le fichier [`Hecke/FltRoute.lean`](../hecke_lean/Hecke/FltRoute.lean) du lake compagnon porte la version formelle de la route. Lisons ce qu'il déclare — et comptons les `sorry`, pour savoir si la preuve est complète :

In [6]:
from pathlib import Path

def trouve_lake():
    import os
    starts = [Path.cwd()] + ([Path(os.environ["LANGLANDS_LAKE_ROOT"])] if os.environ.get("LANGLANDS_LAKE_ROOT") else [])
    for start in starts:
        for cand in (start, *start.parents):
            p = cand / "MyIA.AI.Notebooks/SymbolicAI/Lean/hecke_lean"
            if p.is_dir():
                return p
    raise FileNotFoundError("hecke_lean introuvable")

lake = trouve_lake()
flt = lake / "Hecke" / "FltRoute.lean"
lignes = flt.read_text(encoding="utf-8").splitlines()
print(f"{flt.relative_to(lake)} : {len(lignes)} lignes")
print()
print(f"{'ligne':>5} | declaration")
print("-" * 60)
for i, ligne in enumerate(lignes, 1):
    s = ligne.strip()
    if s.startswith(("theorem ", "def ", "lemma ")):
        nom = s.split("(")[0].split(":")[0].strip()
        print(f"{i:>5} | {nom}")
print()
nb_sorry = sum("sorry" in l for l in lignes)
print(f"occurrences de 'sorry' dans le fichier : {nb_sorry}")
print()
print("Le lake demontre : l'indice 3, S_2(Gamma_0(2)) = 0, l'assemblage final.")
print("Le lake admet (hypotheses explicites de flt_of_full_route) : Wiles, Ribet.")

Hecke\FltRoute.lean : 401 lignes

ligne | declaration
------------------------------------------------------------
   82 | theorem sq_odd_eq_one_mod_eight {k
   96 | theorem odd_pow_congr_self_mod_eight {x p
  116 | theorem frey_congr_mod_eight {a b c p
  155 | def freyCurve
  174 | def firstColMod2
  177 | lemma det_eq_one_mod2
  185 | lemma firstColMod2_ne_zero
  194 | lemma gamma0_two_diag_eq_one {h
  201 | lemma firstColMod2_mul_mem
  214 | def cosetToProj
  226 | lemma cosetToProj_injective
  243 | lemma cosetToProj_surjective
  256 | theorem gamma0_two_index_eq_three
  284 | def CuspForm.norm [ℋ.HasDetPlusMinusOne] [CuspFormClass F 𝒢 k]
  302 | lemma CuspForm.coe_norm_eq_coe_modularFormNorm [ℋ.HasDetPlusMinusOne] [CuspFormClass F 𝒢 k]
  305 | lemma CuspForm.norm_eq_zero_iff [ℋ.HasDetPlusMinusOne] [CuspFormClass F 𝒢 k]
  315 | lemma coe_Gamma_one_eq_SL
  320 | lemma cuspForm_eq_zero_of_subgroup_eq {Γ₁ Γ₂
  327 | theorem s6_levelOne_eq_zero
  332 | theorem s6_levelOne_eq_zero' {k
 

**Lecture — la symétrie des deux mondes.** Ce carnet **calcule** ce que le lake **démontre** : l'indice 3 sort de l'énumération des cosets ici, de la bijection `cosetToProj` là ; le discriminant $16(abc)^{2p}$ sort des invariants $b$ ici, la courbe `freyCurve` est une `WeierstrassCurve` là. Et le lake est honnête sur ses limites : `flt_of_full_route` prend en **hypothèses** les étapes 2 et 5 de la route (la modularité de Wiles et la descente de Ribet) — il ne prétend pas avoir formalisé la preuve de Fermat, il en formalise l'*architecture*, avec les deux théorèmes du XXᵉ siècle explicitement admis. Zéro `sorry` : ce qui est démontré l'est complètement.

## 6. Épilogue — ce que la chaîne a changé

La preuve de Wiles (1995, avec Richard Taylor sur la dernière marche) clôt 358 ans de recherche — mais l'outil a survécu au théorème. La conjecture de modularité, devenue théorème complet (Breuil–Conrad–Diamond–Taylor, 2001), est aujourd'hui une **infrastructure** : c'est elle qui rend calculable la table $a_p$ du carnet 01, elle qui relie les courbes elliptiques aux formes modulaires dans le programme de Langlands, elle encore qui donne aux fonctions $L$ leurs coefficients — dont les zéros peuplent le carnet [`Serre100/07`](../Serre100/07-zeros-fonctions-l-gaps-gue.ipynb).

La chaîne de Frey–Serre–Ribet–Wiles est aussi un **plaidoyer pour les ponts** : personne n'attaquait $x^p + y^p = z^p$ en front, mais relier deux mondes a priori étrangers — comptage de points et séries formelles — a suffi. C'est la direction que Serre décrivait comme orthogonale à la marée montante (carnet 01, introduction) : les voies surprenantes plutôt que le siège frontal.

Dernière face calculée — la courbe jouet $y^2 = x(x-9)(x+16)$ du triplet $(3,4,5)$ porte, comme toute courbe elliptique, ses traces de Frobenius ; la **borne de Hasse** $|a_p| \leq 2\sqrt{p}$ est la face géométrique d'une propriété des coefficients des formes modulaires (conjecture de Ramanujan–Petersson, démontrée par Deligne) :

In [7]:
def points_affines_frey_toy(p):
    """Solutions affines de y^2 = x(x - 9)(x + 16) dans F_p x F_p."""
    return sum(1 for x in range(p) for y in range(p)
               if (y * y - x * (x - 9) * (x + 16)) % p == 0)

def a_p_toy(p):
    """a_p = p + 1 - #E(F_p) pour la courbe jouet du triplet (3, 4, 5)."""
    return p + 1 - (points_affines_frey_toy(p) + 1)

premiers = (5, 7, 11, 13, 17, 19)
print(f"{'p':>3} | {'a_p':>4} | {'2*sqrt(p) arrondi bas':>22} | Hasse")
ok = True
for p in premiers:
    ap = a_p_toy(p)
    conforme = ap * ap <= 4 * p
    ok = ok and conforme
    print(f"{p:>3} | {ap:>4} | {math.isqrt(4 * p):>22} | {'ok' if conforme else 'ECART'}")
assert ok
print()
print("Toutes les traces respectent |a_p|^2 <= 4p : la borne de Hasse, verifiee.")

  p |  a_p |  2*sqrt(p) arrondi bas | Hasse
  5 |    1 |                      4 | ok
  7 |    0 |                      5 | ok
 11 |    4 |                      6 | ok
 13 |   -2 |                      7 | ok
 17 |    2 |                      8 | ok
 19 |   -4 |                      8 | ok

Toutes les traces respectent |a_p|^2 <= 4p : la borne de Hasse, verifiee.


**Lecture.** $a_p^2 \leq 4p$ pour chaque premier testé — la borne de Hasse, version calculée. Pour une forme parabolique de poids 2, le même encadrement est la conjecture de Ramanujan–Petersson (Deligne, 1974) : la modularité fait des traces de Frobenius d'une courbe elliptique les coefficients d'une forme — et leurs tailles obéissent aux mêmes lois.

## 7. Exercices

Trois exercices suivent le fil du carnet ; chaque stub s'exécute sans erreur (convention C.1 : le carnet reste exécutable de bout en bout), la solution attendue est votre code.

### Exercice 1 — le théorème vérifié sur un borné

L'énoncé de Fermat pour $n = 3$ : écrire une recherche exhaustive qui confirme l'absence de solution $a^3 + b^3 = c^3$ sur $1 \leq a, b, c \leq \text{borne}$.

In [8]:
def solutions_flt3(borne):
    """Liste des solutions de a^3 + b^3 = c^3 avec 1 <= a, b, c <= borne.

    TODO etudiant : renvoyer la liste (vide en principe) des triplets trouves.
    Indice : double boucle sur (a, b) ; poser c = round((a**3 + b**3) ** (1/3))
    puis verifier c**3 == a**3 + b**3.
    """
    # Etape 1 : boucle sur a puis sur b dans [1, borne]
    # Etape 2 : extraire la racine cubite entiere candidate et verifier l'egalite
    # Etape 3 : accumuler les triplets (a, b, c) solutions
    return None  # TODO etudiant

print("Exercice a completer : solutions_flt3")

Exercice a completer : solutions_flt3


### Exercice 2 — le discriminant de Frey pour un triplet arbitraire

La fonction `invariants_frey(A, B)` de la section 2 calcule $(c_4, c_6, \Delta)$ pour $y^2 = x(x-A)(x+B)$. Relier les deux : pour un triplet $(a, b)$ et un exposant $n$, renvoyer le discriminant de la courbe de Frey associée — et vérifier la forme fermée $16\,(a^n b^n (a^n + b^n))^2$ sur $(3, 4, 2)$.

In [9]:
def discriminant_frey(a, b, n):
    """Discriminant de y^2 = x(x - a^n)(x + b^n), par la forme fermee.

    TODO etudiant : renvoyer 16 * (a^n * b^n * (a^n + b^n))^2.
    Indice : une seule expression, entiere exacte.
    """
    # Etape 1 : poser A = a**n et B = b**n
    # Etape 2 : appliquer la formule Delta = 16 * (A * B * (A + B))^2
    return None  # TODO etudiant

print("Exercice a completer : discriminant_frey")

Exercice a completer : discriminant_frey


### Exercice 3 — la borne de Hasse poussée à $p = 23$

La table de la section 6 s'arrête à $p = 19$. L'étendre : calculer $a_{23}$ de la courbe jouet $y^2 = x(x-9)(x+16)$ et vérifier $a_{23}^2 \leq 92$.

In [10]:
def a23_et_borne():
    """Couple (a_23, conforme) pour la courbe jouet, conforme = (a_23**2 <= 4*23).

    TODO etudiant : reutiliser points_affines_frey_toy et le schema de la section 6.
    Indice : a_p = p + 1 - (points affines + 1), puis comparer a_23^2 a 4*23.
    """
    # Etape 1 : compter les solutions affines modulo 23
    # Etape 2 : en tirer a_23 puis le booleen a_23**2 <= 4 * 23
    return None  # TODO etudiant

print("Exercice a completer : a23_et_borne")

Exercice a completer : a23_et_borne


## Conclusion

Ce que ce carnet a distillé — la chaîne de modularité, maillon par maillon :

| Maillon | Objet | Où il vit |
|---|---|---|
| Réduction de Fermat | il reste $n \in \{3, 4\}$ et les premiers $p \geq 5$ | calculé (section 1) |
| Courbe de Frey | $y^2 = x(x - a^p)(x + b^p)$, $\Delta = 16(abc)^{2p}$ | calculé (section 2) |
| Modularité | $E \leftrightarrow f \in S_2(\Gamma_0(N))$, $a_p = c_p$ | calculé sur 11a1 (section 3) |
| Descente de Ribet | la forme descend au niveau 2 | admis (théorème 1986) |
| Niveau 2 vide | $[SL_2(\mathbb{Z}) : \Gamma_0(2)] = 3$, $S_2(\Gamma_0(2)) = 0$ | indice calculé, vide démontré (section 4) |
| Assemblage | pas de forme ⇒ pas de contre-exemple | démontré dans le lake (section 5) |

La suite de la série : les fonctions $L$ et leurs zéros ([`Serre100/07`](../Serre100/07-zeros-fonctions-l-gaps-gue.ipynb)), et le point de vue de Langlands lui-même — la correspondance comme **réciproque** plutôt qu'analogie.

## Ressources

- Jean-Pierre Serre, *A Course in Arithmetic*, Springer, ch. VII — le socle des carnets 01 et 03.
- Diamond & Shurman, *A First Course in Modular Forms*, Springer — la référence à archiver pour la série (cf. [README de la série](README.md)).
- G. Frey (1986), « Links between stable elliptic curves and certain Diophantine equations » ; K. Ribet (1990), « On modular representations of Gal(Q̄/Q) arising from modular forms » ; A. Wiles (1995), « Modular elliptic curves and Fermat's Last Theorem », *Annals of Mathematics*.
- Dans le dépôt : [carnet 01](01-formes-modulaires-sl2z-hecke.ipynb) (socle, Hecke, pont 11a1) · [carnet 02](02-monstrous-moonshine-invariant-j.ipynb) (invariant $j$, Monstre) · lake [`hecke_lean/`](../hecke_lean/README.md) (`Hecke/FltRoute.lean`, la route formelle) · [`Serre100/07`](../Serre100/07-zeros-fonctions-l-gaps-gue.ipynb) (zéros des fonctions $L$).
- Sources vidéo de la série (Serre, Serre–Connes, Borcherds) : transcriptions timestampées référencées dans le [README de la série](README.md).